# Preprocesamiento de la serie temporal para modelado

Este notebook parte de la serie mensual ya reconstruida y **no entrena modelos**.

Su objetivo es dejar una salida limpia, auditada y lista para el notebook de modelación.

## Reglas principales

- Lee `serie_mensual_consumo_reconstruida.parquet`.
- Recupera `ciclo` y `clase_servicio` desde los históricos `historico_*.parquet`.
- Identifica **Alumbrado Público** con la regla de negocio:
  - `ciclo == 15`
  - `clase_servicio == "AP"`
- Excluye de la serie de clientes todos los NIU identificados como Alumbrado Público.
- No rellena meses faltantes con cero.
- No elimina consumos cero, negativos o extremos automáticamente: los **audita y marca** para que la decisión se tome en el notebook de modelación.
- Valida unicidad `NIU-periodo`, rango temporal, cobertura mensual, cobertura por cliente y procedencia del consumo.
- Guarda una nueva salida Parquet independiente del archivo reconstruido original.

In [23]:
# ============================================================
# 1. LIBRERÍAS Y RUTAS
# ============================================================

from pathlib import Path
import re
import gc

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

# ------------------------------------------------------------
# Ajustar solo esta ruta si el proyecto cambia de ubicación
# ------------------------------------------------------------
BASE_DIR = Path(r"C:\Users\Home\Documents\Datos Ebsa")

SERIE_DIR = BASE_DIR / "Serie_tiempo_consumo" / "salidas"
PROCESADO_DIR = BASE_DIR / "Procesado"

# Nueva carpeta: no sobrescribe la reconstrucción original
PREPROC_DIR = BASE_DIR / "Serie_tiempo_consumo" / "preprocesamiento_modelo"
PREPROC_DIR.mkdir(parents=True, exist_ok=True)

RUTA_SERIE = SERIE_DIR / "serie_mensual_consumo_reconstruida.parquet"

RUTA_SALIDA = PREPROC_DIR / "serie_mensual_modelado_preprocesada.parquet"
RUTA_AP_EXCLUIDOS = PREPROC_DIR / "nius_alumbrado_publico_excluidos.parquet"
RUTA_AUDITORIA = PREPROC_DIR / "auditoria_preprocesamiento_modelado.csv"
RUTA_COBERTURA_MENSUAL = PREPROC_DIR / "auditoria_cobertura_mensual.csv"

print("Serie de entrada :", RUTA_SERIE)
print("Históricos       :", PROCESADO_DIR)
print("Salida principal :", RUTA_SALIDA)

Serie de entrada : C:\Users\Home\Documents\Datos Ebsa\Serie_tiempo_consumo\salidas\serie_mensual_consumo_reconstruida.parquet
Históricos       : C:\Users\Home\Documents\Datos Ebsa\Procesado
Salida principal : C:\Users\Home\Documents\Datos Ebsa\Serie_tiempo_consumo\preprocesamiento_modelo\serie_mensual_modelado_preprocesada.parquet


In [24]:
# ============================================================
# 2. CARGAR LA SERIE TEMPORAL RECONSTRUIDA
# ============================================================

if not RUTA_SERIE.exists():
    raise FileNotFoundError(
        "No existe la serie reconstruida:\n"
        f"{RUTA_SERIE}"
    )

serie = pd.read_parquet(
    RUTA_SERIE,
    engine="pyarrow"
)

print("Serie cargada")
print(f"Filas      : {len(serie):,}")
print(f"Columnas   : {len(serie.columns):,}")
print(f"NIU únicos : {serie['NIU'].nunique():,}")
print("\nColumnas:")
print(serie.columns.tolist())

Serie cargada
Filas      : 26,890,074
Columnas   : 12
NIU únicos : 592,186

Columnas:
['NIU', 'periodo', 'consumo_kwh_mensual', 'dias_asignados', 'lecturas_que_aportan', 'metodos_usados', 'origen_consumo', 'consumo_imputado', 'candidato_trimestral', 'pct_lecturas_largas', 'pct_saltos_3_meses', 'mediana_dias']


In [25]:
# ============================================================
# 3. NORMALIZAR TIPOS BÁSICOS
# ============================================================

columnas_obligatorias = [
    "NIU",
    "periodo",
    "consumo_kwh_mensual",
]

faltantes = [
    c for c in columnas_obligatorias
    if c not in serie.columns
]

if faltantes:
    raise ValueError(
        "Faltan columnas obligatorias en la serie reconstruida: "
        f"{faltantes}"
    )

serie["NIU"] = (
    serie["NIU"]
    .astype("string")
    .str.strip()
)

serie["periodo"] = pd.to_datetime(
    serie["periodo"],
    errors="coerce"
)

serie["consumo_kwh_mensual"] = pd.to_numeric(
    serie["consumo_kwh_mensual"],
    errors="coerce"
)

# Garantizar orden estable
serie = (
    serie
    .sort_values(["NIU", "periodo"])
    .reset_index(drop=True)
)

print("Tipos normalizados")
print(serie[["NIU", "periodo", "consumo_kwh_mensual"]].dtypes)

Tipos normalizados
NIU                            string
periodo                datetime64[us]
consumo_kwh_mensual           float64
dtype: object


In [26]:
# ============================================================
# 4. VALIDACIONES DE INTEGRIDAD ANTES DEL FILTRO
# ============================================================

nulos_niu = int(serie["NIU"].isna().sum())
nulos_periodo = int(serie["periodo"].isna().sum())
nulos_consumo = int(serie["consumo_kwh_mensual"].isna().sum())

duplicados_niu_periodo = int(
    serie.duplicated(
        subset=["NIU", "periodo"]
    ).sum()
)

periodos_no_inicio_mes = int(
    (
        serie["periodo"].notna()
        & (serie["periodo"].dt.day != 1)
    ).sum()
)

print("VALIDACIÓN INICIAL")
print("-" * 55)
print(f"Nulos NIU              : {nulos_niu:,}")
print(f"Nulos periodo          : {nulos_periodo:,}")
print(f"Nulos consumo          : {nulos_consumo:,}")
print(f"Duplicados NIU-periodo : {duplicados_niu_periodo:,}")
print(f"Periodo no inicia día 1: {periodos_no_inicio_mes:,}")

if nulos_niu > 0 or nulos_periodo > 0:
    raise ValueError(
        "Hay NIU o periodos nulos. Deben corregirse antes de continuar."
    )

if duplicados_niu_periodo > 0:
    raise ValueError(
        "Existen duplicados NIU-periodo. No se eliminarán automáticamente."
    )

VALIDACIÓN INICIAL
-------------------------------------------------------
Nulos NIU              : 0
Nulos periodo          : 0
Nulos consumo          : 0
Duplicados NIU-periodo : 0
Periodo no inicia día 1: 0


## Recuperación de Ciclo y Clase de Servicio

La serie reconstruida no conserva directamente `ciclo` ni `clase_servicio`. Por eso se recuperan desde los históricos anuales ya procesados.

Para evitar cargar columnas innecesarias, solo se leen:

`NIU`, `periodo`, `ciclo`, `clase_servicio`.

In [27]:
# ============================================================
# 5. LOCALIZAR HISTÓRICOS ANUALES
# ============================================================

patron = re.compile(
    r"^historico_(20\d{2})\.parquet$",
    re.IGNORECASE
)

archivos_historicos = sorted(
    [
        p for p in PROCESADO_DIR.glob("historico_*.parquet")
        if patron.match(p.name)
    ],
    key=lambda p: int(patron.match(p.name).group(1))
)

if not archivos_historicos:
    raise FileNotFoundError(
        "No se encontraron archivos historico_YYYY.parquet en:\n"
        f"{PROCESADO_DIR}"
    )

print(f"Históricos encontrados: {len(archivos_historicos)}")
for p in archivos_historicos:
    print(" •", p.name)

Históricos encontrados: 5
 • historico_2022.parquet
 • historico_2023.parquet
 • historico_2024.parquet
 • historico_2025.parquet
 • historico_2026.parquet


In [28]:
# ============================================================
# 6. CARGAR METADATOS DE CICLO Y CLASE DE SERVICIO
# ============================================================

metadata_partes = []

errores_metadata = []

for ruta in archivos_historicos:
    print(f"Leyendo metadatos: {ruta.name}")

    try:
        # Leer únicamente el esquema; NO cargar todo el Parquet
        columnas_archivo = (
            pq.ParquetFile(ruta)
            .schema_arrow
            .names
        )

        necesarias = [
            c for c in ["NIU", "periodo", "ciclo", "clase_servicio"]
            if c in columnas_archivo
        ]

        if "NIU" not in necesarias:
            raise ValueError("No contiene NIU")

        faltan_clasificacion = [
            c for c in ["ciclo", "clase_servicio"]
            if c not in necesarias
        ]

        if faltan_clasificacion:
            print(
                "   ⚠ No tiene todas las columnas de clasificación: ",
                faltan_clasificacion
            )

        temp = pd.read_parquet(
            ruta,
            columns=necesarias,
            engine="pyarrow"
        )

        # Asegurar columnas para concatenación
        if "periodo" not in temp.columns:
            temp["periodo"] = pd.NaT
        if "ciclo" not in temp.columns:
            temp["ciclo"] = np.nan
        if "clase_servicio" not in temp.columns:
            temp["clase_servicio"] = pd.NA

        temp["archivo_historico"] = ruta.name
        metadata_partes.append(temp)

        del temp
        gc.collect()

    except Exception as e:
        errores_metadata.append(
            {
                "archivo": ruta.name,
                "error": str(e),
            }
        )
        print(f"   ERROR: {type(e).__name__}: {e}")

if not metadata_partes:
    raise ValueError(
        "No fue posible recuperar metadatos de ningún histórico."
    )

metadata = pd.concat(
    metadata_partes,
    ignore_index=True,
    sort=False
)

del metadata_partes
gc.collect()

print("\nMetadatos unidos")
print(f"Filas      : {len(metadata):,}")
print(f"NIU únicos : {metadata['NIU'].nunique():,}")

if errores_metadata:
    print("\n⚠ Archivos con error al leer metadatos:")
    display(pd.DataFrame(errores_metadata))

Leyendo metadatos: historico_2022.parquet
Leyendo metadatos: historico_2023.parquet
Leyendo metadatos: historico_2024.parquet
Leyendo metadatos: historico_2025.parquet
Leyendo metadatos: historico_2026.parquet

Metadatos unidos
Filas      : 20,226,965
NIU únicos : 592,186


In [29]:
# ============================================================
# 7. NORMALIZAR CICLO Y CLASE DE SERVICIO
# ============================================================

metadata["NIU"] = (
    metadata["NIU"]
    .astype("string")
    .str.strip()
)

metadata["periodo"] = pd.to_datetime(
    metadata["periodo"],
    errors="coerce"
)

metadata["ciclo_num"] = pd.to_numeric(
    metadata["ciclo"],
    errors="coerce"
)

metadata["clase_servicio_norm"] = (
    metadata["clase_servicio"]
    .astype("string")
    .str.strip()
    .str.upper()
)

metadata["es_ap_registro"] = (
    metadata["ciclo_num"].eq(15)
    & metadata["clase_servicio_norm"].eq("AP")
)

print("Clasificación normalizada")
print("\nRegistros con ciclo 15:")
print(f"{metadata['ciclo_num'].eq(15).sum():,}")

print("\nRegistros con clase AP:")
print(f"{metadata['clase_servicio_norm'].eq('AP').sum():,}")

print("\nRegistros que cumplen ciclo 15 + clase AP:")
print(f"{metadata['es_ap_registro'].sum():,}")

Clasificación normalizada

Registros con ciclo 15:
1,230

Registros con clase AP:
5,542

Registros que cumplen ciclo 15 + clase AP:
1,176


In [30]:
# ============================================================
# DIAGNOSTICO: DISTRIBUCION DE CLIENTES POR CICLO
# ============================================================

# Ciclo más frecuente por NIU (por si cambia entre periodos)
ciclo_por_niu = (
    metadata
    .dropna(subset=["ciclo_num"])
    .groupby("NIU")["ciclo_num"]
    .agg(lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan)
    .rename("ciclo")
    .reset_index()
)

print(
    f"NIU con ciclo identificado: {len(ciclo_por_niu):,} "
    f"de {metadata['NIU'].nunique():,}"
)

resumen_ciclo = (
    ciclo_por_niu
    .groupby("ciclo")
    .size()
    .rename("n_clientes")
    .reset_index()
    .sort_values("ciclo")
)

resumen_ciclo["pct"] = (
    resumen_ciclo["n_clientes"] / resumen_ciclo["n_clientes"].sum() * 100
)

display(resumen_ciclo)

# Guardar el detalle NIU -> ciclo para poder cruzarlo luego
# contra candidato_trimestral y contra el perfil P0-P3 del modelo
RUTA_NIU_CICLO = PREPROC_DIR / "niu_ciclo.parquet"

ciclo_por_niu.to_parquet(
    RUTA_NIU_CICLO,
    index=False,
    engine="pyarrow",
)

print("\nGuardado:", RUTA_NIU_CICLO)

NIU con ciclo identificado: 591,409 de 592,186


,ciclo,n_clientes,pct
0,0.0,95087,16.078044
1,1.0,79729,13.481195
2,2.0,68802,11.633573
3,3.0,29663,5.015649
4,4.0,17190,2.906618
5,5.0,11311,1.912551
6,6.0,10789,1.824287
7,7.0,21018,3.553886
8,9.0,28036,4.740543
9,10.0,30769,5.202660



Guardado: C:\Users\Home\Documents\Datos Ebsa\Serie_tiempo_consumo\preprocesamiento_modelo\niu_ciclo.parquet


In [31]:
# ============================================================
# 8. AUDITAR CONSISTENCIA DE LA REGLA DE ALUMBRADO PÚBLICO
# ============================================================

solo_ciclo_15 = (
    metadata["ciclo_num"].eq(15)
    & ~metadata["clase_servicio_norm"].eq("AP")
)

solo_clase_ap = (
    metadata["clase_servicio_norm"].eq("AP")
    & ~metadata["ciclo_num"].eq(15)
)

print("AUDITORÍA ALUMBRADO PÚBLICO")
print("-" * 55)
print(
    "Ciclo 15 + clase AP       :",
    f"{metadata['es_ap_registro'].sum():,} registros"
)
print(
    "Ciclo 15 pero clase != AP :",
    f"{solo_ciclo_15.sum():,} registros"
)
print(
    "Clase AP pero ciclo != 15 :",
    f"{solo_clase_ap.sum():,} registros"
)

# NIU que en algún momento cumplen exactamente la regla de negocio
nius_ap = (
    metadata.loc[
        metadata["es_ap_registro"],
        "NIU"
    ]
    .dropna()
    .drop_duplicates()
)

print("\nNIU identificados como Alumbrado Público:")
print(f"{len(nius_ap):,}")

# Validación de NIU con estados mixtos en el histórico
estado_niu = (
    metadata
    .groupby("NIU")
    .agg(
        tiene_ap=("es_ap_registro", "max"),
        tiene_registro_no_ap=("es_ap_registro", lambda s: (~s).any()),
        registros_metadata=("NIU", "size"),
    )
    .reset_index()
)

nius_mixtos = estado_niu[
    estado_niu["tiene_ap"]
    & estado_niu["tiene_registro_no_ap"]
]

print("NIU AP con algún registro histórico no AP:")
print(f"{len(nius_mixtos):,}")

if len(nius_mixtos) > 0:
    print(
        "\nNota: por defecto se excluye el NIU completo si alguna vez "
        "cumple ciclo=15 y clase=AP, para evitar que Alumbrado Público "
        "entre al universo de clientes del modelo."
    )

AUDITORÍA ALUMBRADO PÚBLICO
-------------------------------------------------------
Ciclo 15 + clase AP       : 1,176 registros
Ciclo 15 pero clase != AP : 0 registros
Clase AP pero ciclo != 15 : 4,366 registros

NIU identificados como Alumbrado Público:
27
NIU AP con algún registro histórico no AP:
27

Nota: por defecto se excluye el NIU completo si alguna vez cumple ciclo=15 y clase=AP, para evitar que Alumbrado Público entre al universo de clientes del modelo.


In [32]:
# ============================================================
# 9. EXCLUIR ALUMBRADO PÚBLICO DEL UNIVERSO DE MODELADO
# ============================================================
# Regla: si un NIU fue identificado como ciclo 15 + clase AP
# en cualquier periodo disponible, se excluye completamente.
# ============================================================

filas_antes = len(serie)
nius_antes = serie["NIU"].nunique()

mask_ap = serie["NIU"].isin(nius_ap)

filas_ap = int(mask_ap.sum())
nius_ap_presentes = int(
    serie.loc[mask_ap, "NIU"].nunique()
)

serie_modelado = (
    serie.loc[~mask_ap]
    .copy()
    .reset_index(drop=True)
)

print("FILTRO ALUMBRADO PÚBLICO")
print("-" * 55)
print(f"Filas antes                  : {filas_antes:,}")
print(f"NIU antes                    : {nius_antes:,}")
print(f"Filas AP excluidas           : {filas_ap:,}")
print(f"NIU AP excluidos             : {nius_ap_presentes:,}")
print(f"Filas después                : {len(serie_modelado):,}")
print(f"NIU después                  : {serie_modelado['NIU'].nunique():,}")

# Validación obligatoria
ap_restantes = int(
    serie_modelado["NIU"].isin(nius_ap).sum()
)

print(f"\nRegistros AP restantes       : {ap_restantes:,}")

if ap_restantes != 0:
    raise ValueError(
        "La exclusión de Alumbrado Público no quedó completa."
    )

FILTRO ALUMBRADO PÚBLICO
-------------------------------------------------------
Filas antes                  : 26,890,074
NIU antes                    : 592,186
Filas AP excluidas           : 1,323
NIU AP excluidos             : 27
Filas después                : 26,888,751
NIU después                  : 592,159

Registros AP restantes       : 0


In [33]:
# ============================================================
# 9b. AGREGAR CICLO Y FLAG DE ZONA RURAL AL UNIVERSO DE MODELADO
# ============================================================
# Ciclos identificados como rurales/trimestrales (cruce ciclo vs
# candidato_trimestral): >= 90% de sus clientes leen cada ~3 meses.
# ============================================================

CICLOS_RURALES = [10, 11, 12, 13, 19, 21, 22, 23, 38]

ciclo_por_niu = (
    metadata
    .dropna(subset=["ciclo_num"])
    .groupby("NIU")["ciclo_num"]
    .agg(lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan)
    .rename("ciclo")
    .reset_index()
)

ciclo_por_niu["es_rural"] = (
    ciclo_por_niu["ciclo"].isin(CICLOS_RURALES)
)

serie_modelado = serie_modelado.merge(
    ciclo_por_niu,
    on="NIU",
    how="left",
)

n_filas_sin_ciclo = int(serie_modelado["ciclo"].isna().sum())
n_niu_sin_ciclo = int(
    serie_modelado.loc[serie_modelado["ciclo"].isna(), "NIU"].nunique()
)

serie_modelado["es_rural"] = (
    serie_modelado["es_rural"].astype("boolean")  # nullable: True / False / <NA>
)

print("CICLO / ZONA RURAL AGREGADOS AL UNIVERSO DE MODELADO")
print("-" * 60)
print(f"Filas sin ciclo identificado : {n_filas_sin_ciclo:,}")
print(f"NIU sin ciclo identificado   : {n_niu_sin_ciclo:,}")

resumen_rural = (
    serie_modelado
    .drop_duplicates("NIU")
    .groupby("es_rural", dropna=False)["NIU"]
    .size()
    .rename("n_clientes")
    .reset_index()
)

resumen_rural["pct"] = (
    resumen_rural["n_clientes"] / resumen_rural["n_clientes"].sum() * 100
)

display(resumen_rural)

CICLO / ZONA RURAL AGREGADOS AL UNIVERSO DE MODELADO
------------------------------------------------------------
Filas sin ciclo identificado : 777
NIU sin ciclo identificado   : 777


,es_rural,n_clientes,pct
0,False,372022,62.824681
1,True,219360,37.044105
2,<NA>,777,0.131215


## Validaciones para modelación

A partir de aquí no se entrena ningún algoritmo. Se valida que la serie resultante sea coherente para que el siguiente notebook pueda encargarse de features, split temporal, backtesting y modelos.

In [34]:
# ============================================================
# 10. LIMPIEZA TÉCNICA FINAL
# ============================================================

# Infinitos no son valores válidos para modelación
inf_consumo = np.isinf(
    serie_modelado["consumo_kwh_mensual"].to_numpy(dtype="float64")
)

n_inf = int(inf_consumo.sum())

if n_inf > 0:
    print(
        f"⚠ Se encontraron {n_inf:,} consumos infinitos. "
        "Se convierten a NaN, no a cero."
    )
    serie_modelado.loc[
        inf_consumo,
        "consumo_kwh_mensual"
    ] = np.nan

# Reducir memoria sin modificar el significado
serie_modelado["consumo_kwh_mensual"] = (
    serie_modelado["consumo_kwh_mensual"]
    .astype("float32")
)

# Flags de calidad: NO eliminan registros
serie_modelado["consumo_es_cero"] = (
    serie_modelado["consumo_kwh_mensual"].eq(0)
)

serie_modelado["consumo_es_negativo"] = (
    serie_modelado["consumo_kwh_mensual"].lt(0)
)

# Umbral descriptivo global. Se marca, no se elimina.
p999 = float(
    serie_modelado["consumo_kwh_mensual"]
    .quantile(0.999)
)

serie_modelado["consumo_extremo_p999"] = (
    serie_modelado["consumo_kwh_mensual"] > p999
)

print("Umbral P99.9 de consumo:")
print(f"{p999:,.3f} kWh")

Umbral P99.9 de consumo:
7,080.000 kWh


In [35]:
# ============================================================
# 11. AUDITORÍA DEL CONSUMO
# ============================================================

consumo = serie_modelado["consumo_kwh_mensual"]

print("AUDITORÍA DE CONSUMO")
print("-" * 55)
print(f"Registros             : {len(consumo):,}")
print(f"Nulos                 : {consumo.isna().sum():,}")
print(f"Cero                  : {consumo.eq(0).sum():,}")
print(f"Negativos             : {consumo.lt(0).sum():,}")
print(f"Mayores a P99.9       : {serie_modelado['consumo_extremo_p999'].sum():,}")

print("\nDistribución:")
display(
    consumo.describe(
        percentiles=[
            .01, .05, .10, .25, .50,
            .75, .90, .95, .99, .995, .999
        ]
    )
)

AUDITORÍA DE CONSUMO
-------------------------------------------------------
Registros             : 26,888,751
Nulos                 : 0
Cero                  : 2,322,084
Negativos             : 0
Mayores a P99.9       : 26,857

Distribución:


count    2.688875e+07
mean     1.241379e+02
std      2.107600e+03
min      0.000000e+00
1%       0.000000e+00
5%       0.000000e+00
10%      6.521739e-01
25%      1.819149e+01
50%      5.586255e+01
75%      1.030000e+02
90%      1.730000e+02
95%      2.581319e+02
99%      9.124891e+02
99.5%    1.680000e+03
99.9%    7.080000e+03
max      7.951290e+05
Name: consumo_kwh_mensual, dtype: float64

In [36]:
# ============================================================
# 12. VALIDACIÓN DE COBERTURA TEMPORAL
# ============================================================

periodo_min = serie_modelado["periodo"].min()
periodo_max = serie_modelado["periodo"].max()

meses = pd.date_range(
    start=periodo_min.to_period("M").to_timestamp(),
    end=periodo_max.to_period("M").to_timestamp(),
    freq="MS"
)

n_nius = serie_modelado["NIU"].nunique()

combinaciones_esperadas = int(
    n_nius * len(meses)
)

combinaciones_observadas = int(
    len(serie_modelado)
)

combinaciones_faltantes = int(
    combinaciones_esperadas - combinaciones_observadas
)

print("COBERTURA TEMPORAL")
print("-" * 55)
print(f"Periodo                  : {periodo_min:%Y-%m} → {periodo_max:%Y-%m}")
print(f"Meses calendario         : {len(meses):,}")
print(f"NIU                      : {n_nius:,}")
print(f"NIU-mes esperados        : {combinaciones_esperadas:,}")
print(f"NIU-mes existentes       : {combinaciones_observadas:,}")
print(f"NIU-mes faltantes        : {combinaciones_faltantes:,}")

if combinaciones_faltantes > 0:
    print(
        "\nLos NIU-mes faltantes NO se rellenan con cero. "
        "El notebook de modelación deberá tratarlos como meses sin observación."
    )

COBERTURA TEMPORAL
-------------------------------------------------------
Periodo                  : 2022-01 → 2026-01
Meses calendario         : 49
NIU                      : 592,159
NIU-mes esperados        : 29,015,791
NIU-mes existentes       : 26,888,751
NIU-mes faltantes        : 2,127,040

Los NIU-mes faltantes NO se rellenan con cero. El notebook de modelación deberá tratarlos como meses sin observación.


In [37]:
# ============================================================
# 13. COBERTURA POR CLIENTE
# ============================================================

meses_por_cliente = (
    serie_modelado
    .groupby("NIU")["periodo"]
    .nunique()
)

print("MESES DISPONIBLES POR CLIENTE")
print("-" * 55)

display(
    meses_por_cliente.describe(
        percentiles=[
            .01, .05, .10, .25, .50,
            .75, .90, .95, .99
        ]
    )
)

print(f"Clientes >= 12 meses: {(meses_por_cliente >= 12).sum():,}")
print(f"Clientes >= 24 meses: {(meses_por_cliente >= 24).sum():,}")
print(f"Clientes con {len(meses)} meses: {(meses_por_cliente == len(meses)).sum():,}")

MESES DISPONIBLES POR CLIENTE
-------------------------------------------------------


count    592159.000000
mean         45.407992
std          10.891665
min           1.000000
1%            1.000000
5%           14.000000
10%          37.000000
25%          49.000000
50%          49.000000
75%          49.000000
90%          49.000000
95%          49.000000
99%          49.000000
max          49.000000
Name: periodo, dtype: float64

Clientes >= 12 meses: 566,130
Clientes >= 24 meses: 550,071
Clientes con 49 meses: 514,078


In [38]:
# ============================================================
# 14. COBERTURA Y CONSUMO POR MES
# ============================================================

aggs = {
    "registros": ("NIU", "size"),
    "niu_unicos": ("NIU", "nunique"),
    "consumo_total_kwh": (
        "consumo_kwh_mensual",
        lambda x: x.sum(min_count=1)
    ),
    "consumo_mediana_kwh": ("consumo_kwh_mensual", "median"),
    "consumos_cero": ("consumo_es_cero", "sum"),
    "consumos_negativos": ("consumo_es_negativo", "sum"),
}

# Si están disponibles, conservar la auditoría de procedencia
if "consumo_imputado" in serie_modelado.columns:
    aggs["consumos_reconstruidos"] = (
        "consumo_imputado",
        "sum"
    )

cobertura_mensual = (
    serie_modelado
    .groupby("periodo")
    .agg(**aggs)
    .reset_index()
    .sort_values("periodo")
)

max_clientes_mes = cobertura_mensual["niu_unicos"].max()

cobertura_mensual["cobertura_vs_max_pct"] = (
    cobertura_mensual["niu_unicos"]
    / max_clientes_mes
    * 100
).round(2)

if "consumos_reconstruidos" in cobertura_mensual.columns:
    cobertura_mensual["pct_reconstruido"] = (
        cobertura_mensual["consumos_reconstruidos"]
        / cobertura_mensual["registros"]
        * 100
    ).round(2)

print("Últimos 15 meses:")
display(cobertura_mensual.tail(15))

Últimos 15 meses:


,periodo,registros,niu_unicos,consumo_total_kwh,consumo_mediana_kwh,consumos_cero,consumos_negativos,consumos_reconstruidos,cobertura_vs_max_pct,pct_reconstruido
34,2024-11-01,561640,561640,69723008.0,56.000000,51362,0,212591,97.28,37.85
35,2024-12-01,562755,562755,69530344.0,55.000000,49978,0,212838,97.47,37.82
36,2025-01-01,564581,564581,72205360.0,57.874294,45443,0,213887,97.79,37.88
37,2025-02-01,565400,565400,67867720.0,53.000000,51927,0,213861,97.93,37.82
38,2025-03-01,566037,566037,66305688.0,53.189472,51837,0,214002,98.04,37.81
39,2025-04-01,567242,567242,70781920.0,56.615383,46764,0,214074,98.25,37.74
40,2025-05-01,568081,568081,68293392.0,55.000000,52924,0,214363,98.39,37.73
41,2025-06-01,569764,569764,70506112.0,56.000000,52499,0,215251,98.68,37.78
42,2025-07-01,570386,570386,69113088.0,53.542728,47560,0,214850,98.79,37.67
43,2025-08-01,570829,570829,70254768.0,55.586208,53714,0,214618,98.87,37.6


In [39]:
# ============================================================
# 15. PROCEDENCIA DEL CONSUMO
# ============================================================

if "origen_consumo" in serie_modelado.columns:
    print("ORIGEN DEL CONSUMO")
    display(
        serie_modelado["origen_consumo"]
        .value_counts(dropna=False)
        .to_frame("registros")
    )

if "consumo_imputado" in serie_modelado.columns:
    print("\nCONSUMO IMPUTADO / RECONSTRUIDO")
    display(
        serie_modelado["consumo_imputado"]
        .value_counts(dropna=False)
        .to_frame("registros")
    )

ORIGEN DEL CONSUMO


,registros
origen_consumo,
observado,16704966
reconstruido_trimestral,10183785



CONSUMO IMPUTADO / RECONSTRUIDO


,registros
consumo_imputado,
False,16704966
True,10183785


In [40]:
# ============================================================
# 16. AUDITORÍA DEL ÚLTIMO MES DISPONIBLE
# ============================================================

ultimo_mes = serie_modelado["periodo"].max()
ultimo = serie_modelado[
    serie_modelado["periodo"].eq(ultimo_mes)
].copy()

print(f"ÚLTIMO MES DISPONIBLE: {ultimo_mes:%Y-%m}")
print("-" * 55)
print(f"Registros   : {len(ultimo):,}")
print(f"NIU únicos  : {ultimo['NIU'].nunique():,}")
print(
    f"Consumo total: "
    f"{ultimo['consumo_kwh_mensual'].sum(min_count=1):,.3f} kWh"
)

if "origen_consumo" in ultimo.columns:
    print("\nOrigen del consumo:")
    display(
        ultimo["origen_consumo"]
        .value_counts(dropna=False)
        .to_frame("registros")
    )

if "consumo_imputado" in ultimo.columns:
    print("\nObservado vs reconstruido:")
    display(
        ultimo["consumo_imputado"]
        .value_counts(dropna=False)
        .to_frame("registros")
    )

print("\nDistribución del consumo:")
display(
    ultimo["consumo_kwh_mensual"].describe(
        percentiles=[.01, .05, .25, .50, .75, .95, .99]
    )
)

ÚLTIMO MES DISPONIBLE: 2026-01
-------------------------------------------------------
Registros   : 577,360
NIU únicos  : 577,360
Consumo total: 63,438,544.000 kWh

Origen del consumo:


,registros
origen_consumo,
observado,363866
reconstruido_trimestral,213494



Observado vs reconstruido:


,registros
consumo_imputado,
False,363866
True,213494



Distribución del consumo:


count    577360.000000
mean        109.876930
std        1785.546265
min           0.000000
1%            0.000000
5%            0.000000
25%           6.685883
50%          41.000000
75%          96.000000
95%         259.000000
99%         899.000000
max      651551.000000
Name: consumo_kwh_mensual, dtype: float64

In [41]:
# ============================================================
# 17. VALIDACIÓN FINAL DE UNICIDAD Y ORDEN
# ============================================================

serie_modelado = (
    serie_modelado
    .sort_values(["NIU", "periodo"])
    .reset_index(drop=True)
)

duplicados_finales = int(
    serie_modelado.duplicated(
        subset=["NIU", "periodo"]
    ).sum()
)

niu_nulos_final = int(serie_modelado["NIU"].isna().sum())
periodo_nulos_final = int(serie_modelado["periodo"].isna().sum())

print("VALIDACIÓN FINAL")
print("-" * 55)
print(f"Duplicados NIU-periodo : {duplicados_finales:,}")
print(f"NIU nulos              : {niu_nulos_final:,}")
print(f"Periodo nulos          : {periodo_nulos_final:,}")
print(f"AP restantes           : {serie_modelado['NIU'].isin(nius_ap).sum():,}")

if duplicados_finales != 0:
    raise ValueError("Persisten duplicados NIU-periodo.")

if niu_nulos_final != 0 or periodo_nulos_final != 0:
    raise ValueError("Persisten NIU o periodos nulos.")

if serie_modelado["NIU"].isin(nius_ap).any():
    raise ValueError("Persisten NIU de Alumbrado Público.")

print("\n✓ La estructura básica está lista para el notebook de modelación.")

VALIDACIÓN FINAL
-------------------------------------------------------
Duplicados NIU-periodo : 0
NIU nulos              : 0
Periodo nulos          : 0
AP restantes           : 0

✓ La estructura básica está lista para el notebook de modelación.


In [42]:
# ============================================================
# 18. RESUMEN DE CALIDAD PARA MODELACIÓN
# ============================================================

resumen_auditoria = pd.DataFrame(
    [
        ["filas_entrada", filas_antes],
        ["nius_entrada", nius_antes],
        ["filas_ap_excluidas", filas_ap],
        ["nius_ap_excluidos", nius_ap_presentes],
        ["filas_salida", len(serie_modelado)],
        ["nius_salida", serie_modelado["NIU"].nunique()],
        ["periodo_min", periodo_min.strftime("%Y-%m")],
        ["periodo_max", periodo_max.strftime("%Y-%m")],
        ["meses_calendario", len(meses)],
        ["duplicados_niu_periodo", duplicados_finales],
        ["nulos_consumo", int(serie_modelado["consumo_kwh_mensual"].isna().sum())],
        ["consumos_cero", int(serie_modelado["consumo_es_cero"].sum())],
        ["consumos_negativos", int(serie_modelado["consumo_es_negativo"].sum())],
        ["consumos_extremos_p999", int(serie_modelado["consumo_extremo_p999"].sum())],
        ["umbral_consumo_p999_kwh", p999],
        ["combinaciones_niu_mes_faltantes", combinaciones_faltantes],
    ],
    columns=["metrica", "valor"]
)

display(resumen_auditoria)

,metrica,valor
0,filas_entrada,26890074
1,nius_entrada,592186
2,filas_ap_excluidas,1323
3,nius_ap_excluidos,27
4,filas_salida,26888751
5,nius_salida,592159
6,periodo_min,2022-01
7,periodo_max,2026-01
8,meses_calendario,49
9,duplicados_niu_periodo,0


In [43]:
# ============================================================
# 19. GUARDAR SALIDAS DEL PREPROCESAMIENTO
# ============================================================

# Principal: dataset que consumirá el notebook de modelación
serie_modelado.to_parquet(
    RUTA_SALIDA,
    index=False,
    engine="pyarrow"
)

# Lista de NIU excluidos por Alumbrado Público
pd.DataFrame({
    "NIU": nius_ap.astype("string")
}).to_parquet(
    RUTA_AP_EXCLUIDOS,
    index=False,
    engine="pyarrow"
)

# Auditorías pequeñas y legibles
resumen_auditoria.to_csv(
    RUTA_AUDITORIA,
    index=False,
    encoding="utf-8-sig"
)

cobertura_mensual.to_csv(
    RUTA_COBERTURA_MENSUAL,
    index=False,
    encoding="utf-8-sig"
)

print("PREPROCESAMIENTO TERMINADO")
print("=" * 60)
print("\nArchivo principal para MODELACIÓN:")
print(RUTA_SALIDA)

print("\nArchivos de auditoría:")
print(" •", RUTA_AP_EXCLUIDOS)
print(" •", RUTA_AUDITORIA)
print(" •", RUTA_COBERTURA_MENSUAL)

PREPROCESAMIENTO TERMINADO

Archivo principal para MODELACIÓN:
C:\Users\Home\Documents\Datos Ebsa\Serie_tiempo_consumo\preprocesamiento_modelo\serie_mensual_modelado_preprocesada.parquet

Archivos de auditoría:
 • C:\Users\Home\Documents\Datos Ebsa\Serie_tiempo_consumo\preprocesamiento_modelo\nius_alumbrado_publico_excluidos.parquet
 • C:\Users\Home\Documents\Datos Ebsa\Serie_tiempo_consumo\preprocesamiento_modelo\auditoria_preprocesamiento_modelado.csv
 • C:\Users\Home\Documents\Datos Ebsa\Serie_tiempo_consumo\preprocesamiento_modelo\auditoria_cobertura_mensual.csv


# Salida esperada

El notebook de modelación deberá leer **únicamente**:

`Serie_tiempo_consumo/preprocesamiento_modelo/serie_mensual_modelado_preprocesada.parquet`

Este notebook no realiza:

- creación de lags;
- creación de targets `t+1`, `t+2`, `t+3`;
- split train/validation/test;
- backtesting;
- entrenamiento de LightGBM u otros modelos.

Eso queda deliberadamente separado para el siguiente notebook.

In [44]:
# ============================================================
# CRUCE: CICLO vs PERIODICIDAD TRIMESTRAL (IDENTIFICAR CICLOS RURALES)
# ============================================================

from pathlib import Path
import pandas as pd

BASE_DIR = Path(r"C:\Users\Home\Documents\Datos Ebsa")

RUTA_NIU_CICLO = (
    BASE_DIR / "Serie_tiempo_consumo" / "preprocesamiento_modelo" / "niu_ciclo.parquet"
)

RUTA_PERFIL_PERIODICIDAD = (
    BASE_DIR / "Serie_tiempo_consumo" / "salidas" / "perfil_periodicidad_clientes.parquet"
)

ciclo_por_niu = pd.read_parquet(RUTA_NIU_CICLO, engine="pyarrow")
perfil_periodicidad = pd.read_parquet(RUTA_PERFIL_PERIODICIDAD, engine="pyarrow")

cruce = ciclo_por_niu.merge(
    perfil_periodicidad[
        [
            "NIU",
            "candidato_trimestral",
            "pct_lecturas_largas",
            "pct_saltos_3_meses",
            "mediana_dias",
        ]
    ],
    on="NIU",
    how="inner",
)

print(f"NIU cruzados: {len(cruce):,} de {len(ciclo_por_niu):,} con ciclo conocido")

resumen_ciclo_rural = (
    cruce
    .groupby("ciclo")
    .agg(
        n_clientes=("NIU", "size"),
        pct_trimestral=("candidato_trimestral", "mean"),
        mediana_dias_lectura=("mediana_dias", "median"),
        pct_lecturas_largas_prom=("pct_lecturas_largas", "mean"),
    )
    .reset_index()
)

resumen_ciclo_rural["pct_trimestral"] *= 100
resumen_ciclo_rural["pct_lecturas_largas_prom"] *= 100

resumen_ciclo_rural = resumen_ciclo_rural.sort_values(
    "pct_trimestral", ascending=False
)

display(resumen_ciclo_rural)

# Ciclos donde la mayoría de clientes son candidatos trimestrales
UMBRAL_RURAL = 50  # % de clientes trimestrales para considerar el ciclo "rural"

ciclos_rurales = resumen_ciclo_rural[
    resumen_ciclo_rural["pct_trimestral"] >= UMBRAL_RURAL
]["ciclo"].tolist()

print(
    f"\nCiclos identificados como predominantemente rurales/trimestrales "
    f"(>= {UMBRAL_RURAL}% de clientes trimestrales):"
)
print(ciclos_rurales)

NIU cruzados: 591,409 de 591,409 con ciclo conocido


,ciclo,n_clientes,pct_trimestral,mediana_dias_lectura,pct_lecturas_largas_prom
15,21.0,15784,99.252408,91.0,99.601031
16,22.0,22474,99.056688,91.0,98.959405
10,11.0,29718,98.734774,91.0,99.431974
12,13.0,17696,98.722875,91.0,99.652578
17,23.0,15904,98.641851,91.0,99.025933
9,10.0,30769,98.335988,91.0,99.155411
14,19.0,51921,98.295487,91.0,99.294458
11,12.0,34697,97.204369,91.0,99.343051
19,38.0,397,90.176322,92.0,98.919860
1,1.0,79729,0.001254,31.0,0.218101



Ciclos identificados como predominantemente rurales/trimestrales (>= 50% de clientes trimestrales):
[21.0, 22.0, 11.0, 13.0, 23.0, 10.0, 19.0, 12.0, 38.0]
